## 🎯 Learning Objectives
* Understand the fundamental architectural differences between BERT and GPT-style models.
* Differentiate between encoder-only (BERT) and decoder-only (GPT) transformer architectures.
* Recognize the implications of these architectural choices on model capabilities and typical use cases.
* Gain practical experience using Hugging Face Transformers to interact with BERT and GPT-style models for their respective primary tasks.


## DL02-L12: BERT and GPT-style Architectures Compared

Welcome to a pivotal lesson where we dissect the two titans that revolutionized Natural Language Processing: BERT (Bidirectional Encoder Representations from Transformers) and GPT (Generative Pre-trained Transformer). While both are built upon the Transformer architecture, their design philosophies, pre-training objectives, and consequently, their strengths and typical applications, diverge significantly.

Imagine you're building two different AI assistants:

1.  **The Comprehension Expert (BERT-style):** This assistant's primary goal is to deeply understand the context of a given text. It reads everything, forwards and backwards, to grasp the full meaning of every word. It's like a seasoned detective analyzing clues from all angles to solve a mystery. This is the essence of **BERT**.

2.  **The Creative Writer (GPT-style):** This assistant's main task is to generate coherent and contextually relevant text, one word at a time. It's like a novelist who, having written the beginning of a story, meticulously crafts the next sentence based on what has already been written, without knowing what the *end* of the story will be. This is the essence of **GPT**.

### BERT: The Encoder-Only, Bidirectional Powerhouse

BERT, introduced by Google in 2018, is an **encoder-only** Transformer model. Its defining characteristic is its **bidirectional attention mechanism**. This means that when BERT processes a word, it considers the context from both the words *before* it and the words *after* it simultaneously. This deep, contextual understanding is achieved through its innovative pre-training objectives:

*   **Masked Language Modeling (MLM):** BERT randomly masks out 15% of the tokens in a sequence and then tries to predict the original masked tokens based on the context provided by the unmasked tokens (both left and right). This forces the model to learn rich, bidirectional representations.
*   **Next Sentence Prediction (NSP):** BERT is also trained to predict whether two sentences appear consecutively in the original document. This helps it understand relationships between sentences, crucial for tasks like question answering and natural language inference.

Because of its bidirectional nature, BERT excels at tasks that require a deep understanding of the input text, such as text classification, sentiment analysis, named entity recognition, and question answering.

### GPT: The Decoder-Only, Unidirectional Generator

GPT, pioneered by OpenAI, represents a family of **decoder-only** Transformer models (e.g., GPT, GPT-2, GPT-3, GPT-4, Llama, etc.). Its core principle is **unidirectional (or causal) attention**. When GPT generates a word, it can only attend to the words that have *already been generated* (or are to its left in the input sequence). It cannot 


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# --- 1. BERT-style Model: Masked Language Modeling (Encoder-only) ---
print("\n--- Demonstrating BERT (Encoder-only) for Masked Language Modeling ---")

# Load pre-trained BERT tokenizer and model
# We'll use a smaller, base model for demonstration purposes.
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
bert_model.eval() # Set model to evaluation mode

# Example sentence with a masked token
text_bert = "The capital of France is [MASK]."

# Tokenize the input and find the mask token's index
input_bert = bert_tokenizer(text_bert, return_tensors="pt")
mask_token_index = torch.where(input_bert["input_ids"] == bert_tokenizer.mask_token_id)[1]

# Move to GPU if available
if torch.cuda.is_available():
    bert_model.to('cuda')
    input_bert = {k: v.to('cuda') for k, v in input_bert.items()}

# Get predictions for the masked token
with torch.no_grad():
    outputs_bert = bert_model(**input_bert)
    predictions = outputs_bert.logits

# Get the top 5 predicted tokens for the mask
predicted_token_ids = torch.topk(predictions[0, mask_token_index, :], 5).indices.tolist()
predicted_tokens = bert_tokenizer.decode(predicted_token_ids)

print(f"Input (BERT): {text_bert}")
print(f"Top 5 predictions for [MASK]: {predicted_tokens}")

# --- 2. GPT-style Model: Text Generation (Decoder-only) ---
print("\n--- Demonstrating GPT (Decoder-only) for Text Generation ---")

# Load pre-trained GPT-2 tokenizer and model
# GPT-2 is a classic example of a decoder-only model.
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt_model = AutoModelForCausalLM.from_pretrained("gpt2")
gpt_model.eval() # Set model to evaluation mode

# GPT models typically don't have a padding token by default, or it's not set.
# For generation, it's good practice to set a pad_token_id if not present,
# especially for batch generation, though not strictly necessary for single sequence here.
if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token # Use EOS token as pad token

# Input prompt for generation
text_gpt = "Once upon a time, in a land far, far away, there was a dragon who"

# Tokenize the input
input_gpt = gpt_tokenizer(text_gpt, return_tensors="pt")

# Move to GPU if available
if torch.cuda.is_available():
    gpt_model.to('cuda')
    input_gpt = {k: v.to('cuda') for k, v in input_gpt.items()}

# Generate text using the model
# We'll generate a short sequence for demonstration
with torch.no_grad():
    generated_ids = gpt_model.generate(
        input_gpt["input_ids"],
        max_new_tokens=30, # Generate up to 30 new tokens
        num_beams=5,       # Use beam search for better quality
        no_repeat_ngram_size=2, # Avoid repeating n-grams
        early_stopping=True, # Stop if all beam hypotheses have met EOS
        pad_token_id=gpt_tokenizer.eos_token_id # Important for generation
    )

# Decode the generated text
generated_text = gpt_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(f"Input (GPT): {text_gpt}")
print(f"Generated text: {generated_text}")


### Interpreting the Output and Architectural Implications

The code above vividly demonstrates the core functionalities and architectural differences between BERT and GPT-style models.

#### BERT Output Interpretation:

For the BERT example, the input was "The capital of France is [MASK]." The model successfully predicted "paris" as the top candidate for the masked token. This is a direct result of BERT's **bidirectional attention** and **Masked Language Modeling (MLM)** pre-training objective. BERT was able to look at "The capital of France is" *and* the implicit end of the sentence to infer the most probable word for the mask. It understands the full context to fill in the blank, making it highly effective for tasks requiring deep contextual understanding, such as:

*   **Text Classification:** Categorizing documents (e.g., spam detection, sentiment analysis).
*   **Named Entity Recognition (NER):** Identifying entities like names, locations, organizations.
*   **Question Answering (QA):** Extracting answers from a given text.
*   **Semantic Search:** Finding documents semantically similar to a query.

#### GPT Output Interpretation:

For the GPT example, given the prompt "Once upon a time, in a land far, far away, there was a dragon who", the model continued the story in a coherent and creative manner. This showcases GPT's strength in **autoregressive text generation**, a direct consequence of its **unidirectional (causal) attention** and **decoder-only architecture**. It predicts the next token based *only* on the preceding tokens, building the sequence word by word. This makes GPT-style models ideal for:

*   **Content Generation:** Writing articles, stories, code, marketing copy.
*   **Summarization:** Condensing long texts into shorter versions (often abstractive).
*   **Chatbots and Conversational AI:** Generating human-like responses in dialogue systems.
*   **Code Generation:** Assisting developers by generating code snippets or completing functions.

#### Performance Trade-offs and Use Cases:

| Feature             | BERT-style (Encoder-only)                               | GPT-style (Decoder-only)                                  |
| :------------------ | :------------------------------------------------------ | :-------------------------------------------------------- |
| **Attention**       | Bidirectional (attends to left and right context)       | Unidirectional/Causal (attends only to left context)      |
| **Pre-training**    | Masked Language Modeling, Next Sentence Prediction      | Autoregressive Language Modeling (predict next token)     |
| **Primary Task**    | Understanding, Classification, Token-level prediction   | Generation, Completion, Creative Writing                  |
| **Strengths**       | Deep contextual understanding, excellent for analysis   | Coherent and fluent text generation, creativity           |
| **Weaknesses**      | Not designed for text generation                        | Less optimal for pure comprehension/classification tasks  |
| **Typical Use Cases** | Sentiment analysis, QA, NER, text classification, search | Chatbots, content creation, summarization, code generation |

It's important to note that while these are the foundational distinctions, the lines can blur. For instance, GPT-style models can be fine-tuned for classification by appending a classification head and using the final token's embedding. Similarly, techniques exist to adapt BERT for generation, though it's not its native strength. Furthermore, **Encoder-Decoder architectures** (like T5, BART, NLLB) combine both components, leveraging the encoder for understanding the input and the decoder for generating the output, making them highly effective for sequence-to-sequence tasks like machine translation and abstractive summarization.

As of 2026, the landscape continues to evolve with larger, more efficient, and specialized models, but the fundamental principles of bidirectional vs. unidirectional attention, and encoder-only vs. decoder-only architectures, remain cornerstones of modern NLP. Understanding these differences is crucial for selecting the right model for your specific task.


### Resources

*   **Hugging Face Transformers Library:** The go-to library for working with pre-trained Transformer models. Explore their extensive documentation for various models and tasks.
    *   [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/index)
    *   [Hugging Face Models Hub](https://huggingface.co/models)

*   **Original BERT Paper:** "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding" by Devlin et al. (2018).
    *   [arXiv Link](https://arxiv.org/abs/1810.04805)

*   **Original GPT Papers / OpenAI Blog Posts:**
    *   **GPT-1:** "Improving Language Understanding by Generative Pre-Training" by Radford et al. (2018) - [OpenAI Blog Post](https://openai.com/blog/language-unsupervised/)
    *   **GPT-2:** "Language Models are Unsupervised Multitask Learners" by Radford et al. (2019) - [OpenAI Blog Post](https://openai.com/blog/better-language-models/)
    *   **GPT-3:** "Language Models are Few-Shot Learners" by Brown et al. (2020) - [arXiv Link](https://arxiv.org/abs/2005.14165)

*   **PyTorch Documentation:** For general deep learning concepts and tensor operations.
    *   [PyTorch Official Documentation](https://pytorch.org/docs/stable/index.html)

*   **Google AI Blog:** Often features insights and updates on Transformer research.
    *   [Google AI Blog](https://ai.googleblog.com/)
